This is to create a tier system for applicants

Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
PROC = ROOT / "data" / "processed"
sys.path.insert(0, str(ROOT))

from src.rating import expected_cost, coefs, COVERAGES

freq = pd.read_csv(PROC / "frequency_data.csv").rename(columns={'class': 'class_year'})

LAE_RATIO = 8.5 / 66.2          # NAIC 2024 IEE, Homeowners Multiple Peril
V         = (18.1 + 2.5) / 100  # variable expense %: selling + taxes, licenses, fees
Q_T       = 0.05                # target underwriting profit %
GEN_EXP   = 0.051               # NAIC general expense % -> fixed expense per exposure

students = freq.drop_duplicates('student_id')

# expected loss + LAE, per policy per coverage
L_c = (students.apply(lambda r: expected_cost(r, detail=True)
                                  .set_index('coverage').pure_premium,
                      axis=1)[COVERAGES] * (1 + LAE_RATIO))

# flat fixed expense per coverage, set off that coverage's mean
E_F_c = GEN_EXP * L_c.mean()

# gross premium per policy per coverage
R_c = (L_c + E_F_c) / (1 - V - Q_T)

L = L_c.sum(axis=1)
R = R_c.sum(axis=1)

print(pd.DataFrame({'loss + LAE': L_c.mean(),
                    'fixed exp':  E_F_c,
                    'gross prem': R_c.mean()}).round(2).to_string())
print("\npolicies:", len(R), " mean gross premium:", round(R.mean(), 2))

Now to make a histogram of all our gross premiums to get some initial analysis

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(R, bins=60, color='#15803D', edgecolor='white', linewidth=0.4)

ax.axvline(R.mean(), color='#0F1A13', linewidth=1.5, linestyle='--')
ax.annotate(f'mean  ${R.mean():,.0f}', xy=(R.mean(), ax.get_ylim()[1] * 0.93),
            xytext=(8, 0), textcoords='offset points', fontsize=10, color='#0F1A13')

ax.set_xlabel('Indicated gross premium ($)')
ax.set_ylabel('Policyholders')
ax.set_title('Distribution of indicated gross premium, 10,000 policyholders')
ax.grid(axis='y', color='#D3E0D7', linewidth=0.8)
ax.set_axisbelow(True)
for s in ('top', 'right', 'left'):
    ax.spines[s].set_visible(False)
ax.spines['bottom'].set_color('#C6D4CB')

plt.tight_layout()
plt.show()

This histogram shows a clear split in gross premium among 2 populations. based on initial testing we think that this is caused by greek vs non-greek applicants but further analysis is required

In [ ]:
R.groupby(students.greek).agg(['count', 'min', 'mean', 'max']).round(0)

In [ ]:
is_greek = students.greek == 'Greek'
bins     = np.linspace(R.min(), R.max(), 60)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist([R[~is_greek], R[is_greek]], bins=bins, stacked=True,
        color=['#15803D', '#0F1A13'], label=['Non-Greek', 'Greek'],
        edgecolor='white', linewidth=0.4)

top = ax.get_ylim()[1]
for m, c in [(R[~is_greek].mean(), '#15803D'), (R[is_greek].mean(), '#0F1A13')]:
    ax.axvline(m, color=c, linewidth=1.5, linestyle='--')
    ax.annotate(f'${m:,.0f}', xy=(m, top * 0.93), xytext=(6, 0),
                textcoords='offset points', fontsize=10, color='#0F1A13')

ax.legend(frameon=False)
ax.set_xlabel('Indicated gross premium ($)')
ax.set_ylabel('Policyholders')
ax.set_title('Indicated gross premium by Greek status')
ax.grid(axis='y', color='#D3E0D7', linewidth=0.8)
ax.set_axisbelow(True)
for s in ('top', 'right', 'left'):
    ax.spines[s].set_visible(False)
ax.spines['bottom'].set_color('#C6D4CB')

plt.tight_layout()
plt.show()

Putting these 2 histograms together the story becomes strikingly clear that we need to seperate greek life and non greek life applicants. And it is also interesting the the non-greek historgram has a right sided tail.

When analyzing our premium algorithm, it was noted that class year offered significant changes in gross premium so we now split find the mean gross premium for each "class year" for both the greek and non greek populations, which from now on will be split

In [ ]:
ORDER = ['Freshman', 'Sophomore', 'Junior', 'Senior', 'Grad student']

def by_coverage(key, order=None):
    t = pd.concat({g: R_c[students.greek == g]
                      .groupby(key[students.greek == g], observed=True).mean()
                   for g in ['Greek', 'Non-greek']})
    if order is not None:
        t = t.reindex(order, level=1)
    t['Total policy'] = t.sum(axis=1)
    return t.round(2)


mean_prem = by_coverage(students.class_year, ORDER)
mean_prem

In [ ]:
gpa_band = pd.cut(students.gpa, bins=np.arange(0, 4.5, 0.5), right=False)

gpa_prem = by_coverage(gpa_band)
gpa_prem

In [ ]:
spr_prem = by_coverage(students.sprinklered)
spr_prem

In [ ]:
oc_prem = by_coverage(students.off_campus)
oc_prem

Since the rating tier is per policy we analyze the affect on the total policy gross premium(last colomn) and we will then determine the base rates for each coverage seperately.
Based on the assumption above, evidence suggest we choose our tiers based on on/off campus. And although it it harder to see, the class year has a significant jump for undergraduate and graduate students so we will see what affect grouping by undergraduate/graduate and on/off campus has on average gross premium for each catagory

In [ ]:
level = pd.Series(np.where(students.class_year == 'Grad student',
                           'Graduate', 'Undergraduate'),
                  index=students.index, name='level')

cells = (R.groupby([students.greek, level, students.off_campus], observed=True)
           .mean().unstack('off_campus')
           .reindex(['Undergraduate', 'Graduate'], level='level')
           .round(2))

greek_tbl    = cells.loc['Greek']
nongreek_tbl = cells.loc['Non-greek']

print("Greek\n",      greek_tbl.to_string(),    "\n")
print("Non-Greek\n",  nongreek_tbl.to_string())

This gives evidence that grouping by undergraduate/graduate had significant effects
These now give the indicated base rate for each greek vs non greek applicant. We will split our population into Greek and Non-Greek, and the rating tiers for each will be as follows
Preferred-plus(Undergraduate, On-campus), Preferred(Undergraduate, Off-Campus), Standard(Graduate, On-campus) and Non-Standard(Graduate, Off-campus)

Now to find the "Base Rate" for each coverage based on the rating tier determined above, we average the gross premium for each individual in the selected tiers

In [ ]:
TIERS = ['Preferred-Plus', 'Preferred', 'Standard', 'Non-Standard']

tier = pd.Series(np.select(
    [(students.class_year != 'Grad student') & (students.off_campus == 'On campus'),
     (students.class_year != 'Grad student') & (students.off_campus == 'Off campus'),
     (students.class_year == 'Grad student') & (students.off_campus == 'On campus')],
    ['Preferred-Plus', 'Preferred', 'Standard'],
    default='Non-Standard'),
    index=students.index, name='tier')

base_rates = by_coverage(tier, TIERS)
base_rates

These give the base rate for each coverage, now to answer the the question of credits and debits for people in each tier we use a factor table to determine the price to charge each individual within the tiers. 

In [ ]:
m   = (students.greek == 'Non-greek') & (tier == 'Preferred-Plus')
rel = R_c[m] / R_c[m].mean()


def tier_factor(key, order=None):
    t = rel.groupby(key[m], observed=True).mean()
    return t.reindex(order) if order is not None else t


UG       = ['Freshman', 'Sophomore', 'Junior', 'Senior']
gpa_band = pd.cut(students.gpa, bins=[0, 1, 2, 3, 4], right=False)

factors = pd.concat({'Sprinklered':    tier_factor(students.sprinklered),
                     'GPA':            tier_factor(gpa_band),
                     'Field of study': tier_factor(students.study),
                     'Class year':     tier_factor(students.class_year, UG)},
                    names=['Variable', 'Level']).round(3)

factors

In [ ]:
m_pref   = (students.greek == 'Non-greek') & (tier == 'Preferred')
rel_pref = R_c[m_pref] / R_c[m_pref].mean()


def tier_factor_pref(key, order=None):
    t = rel_pref.groupby(key[m_pref], observed=True).mean()
    return t.reindex(order) if order is not None else t


dist_band = pd.cut(students.distance_to_campus,
                   bins=[0, 1, 2, 3, 5, 10, np.inf], right=False)

factors_pref = pd.concat(
    {'Sprinklered':        tier_factor_pref(students.sprinklered),
     'GPA':                tier_factor_pref(gpa_band),
     'Field of study':     tier_factor_pref(students.study),
     'Class year':         tier_factor_pref(students.class_year, UG),
     'Distance to campus': tier_factor_pref(dist_band)},
    names=['Variable', 'Level']).round(3)

factors_pref

In [ ]:
UG_TIERS  = {'Preferred-Plus', 'Preferred'}
OFF_TIERS = {'Preferred', 'Non-Standard'}


def factor_table(program, tier_name):
    mk = (students.greek == program) & (tier == tier_name)
    rl = R_c[mk] / R_c[mk].mean()

    def f(key, order=None):
        t = rl.groupby(key[mk], observed=True).mean()
        return t.reindex(order) if order is not None else t

    d = {'Sprinklered':    f(students.sprinklered),
         'GPA':            f(gpa_band),
         'Field of study': f(students.study)}

    if tier_name in UG_TIERS:
        d['Class year'] = f(students.class_year, UG)
    if tier_name in OFF_TIERS:
        d['Distance to campus'] = f(dist_band)

    return (pd.concat(d, names=['Variable', 'Level'])
              .dropna(how='all')
              .round(3))

In [ ]:
CAPTION_STYLE = [{'selector': 'caption',
                  'props': [('font-size', '15px'),
                            ('font-weight', 'bold'),
                            ('text-align', 'left'),
                            ('padding-bottom', '8px')]}]


def show(program, tier_name):
    label = 'Non-Greek' if program == 'Non-greek' else 'Greek'
    return (factor_table(program, tier_name).style
            .set_caption(f"{label} {tier_name}")
            .format("{:.3f}")
            .set_table_styles(CAPTION_STYLE))

In [ ]:
from IPython.display import display

for program in ['Non-greek', 'Greek']:
    for t in TIERS:
        display(show(program, t))

In [ ]:
# modelled pure premium per student per coverage, with the LAE load stripped back out
PP_model = L_c / (1 + LAE_RATIO)

# actual losses, reshaped to the same students x coverage grid
PP_actual = (freq.pivot(index='student_id', columns='coverage', values='total_amount')
                 [COVERAGES]
                 .loc[students.student_id.values])
PP_actual.index = students.index

split = students.training_holdout

rows = []
for pg in ['Non-greek', 'Greek']:
    for sp in ['training', 'holdout']:
        m = (split == sp) & (students.greek == pg)
        for c in COVERAGES:
            mo, ac = PP_model.loc[m, c].mean(), PP_actual.loc[m, c].mean()
            rows.append({'program': pg, 'split': sp, 'coverage': c,
                         'model': mo, 'actual': ac, 'ratio': mo / ac})
        mo = PP_model[m].sum(axis=1).mean()
        ac = PP_actual[m].sum(axis=1).mean()
        rows.append({'program': pg, 'split': sp, 'coverage': 'TOTAL',
                     'model': mo, 'actual': ac, 'ratio': mo / ac})

balance = pd.DataFrame(rows).round({'model': 2, 'actual': 2, 'ratio': 4})
balance